In [ ]:
import sys
from pathlib import Path
import numpy as np

_nb_dir = Path.cwd()
if _nb_dir.name == "homework":
    _topic = _nb_dir.parent
else:
    _topic = _nb_dir / "01-intro-and-kinematics"
if str(_topic) not in sys.path:
    sys.path.insert(0, str(_topic))
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

SO101_URDF = (
    _nb_dir / "assets" / "so101" / "robot.urdf"
    if (_nb_dir / "assets" / "so101" / "robot.urdf").exists()
    else _topic / "homework" / "assets" / "so101" / "robot.urdf"
)

# Problem 1: SO101 Analytical IK (10 pts)

**Trivia.** 
SO-101 arm is our favorite open source robot manipulator. 
We will be working with it a lot, so better get used to it. 
One of the best ways to become familiar with all the dance moves this cutesie has is to compute forward and inverse kinematics by hand, draw some diagrams and remember some trigonometry. 
Same-level tabletop manipulation is the most used setup for the SO-101 in practice (because it is natural and simple). 
For tasks such as pick-and-place, traditional solutions oftentimes rely on the notion of **pre-grasp** pose which is derived from the object we have to manipulate. 
So it is only natural to ask ourselves the question, how do we position our robot, i.e. which angles should robot joints be having, for the end-effector to attain the pre-grasp pose. 
In this task we are to answer this question with various tools.

<video src="assets/so101_cube_pickup.mp4" controls style="max-width: 100%;"></video>

**Task.** 
For simplicity we restrict our pre-grasp poses to the ones where the gripper approaches from above with z-axis pointing straight down in the world frame, and the in-plane rotation (yaw about world Z) is aligned with the object (cube) orientation. This reduces the task to **position (x, y, z) + yaw**, i.e. 4 DOF.

In the class we played around with the SO101 arm and computed forward kinematics and inverse kinematics using numerical solvers implemented in `pytorch_kinematics` library.

<img src="assets/FK-info.png" alt="SO101 forward kinematics" style="max-width: 100%;" />

**Task.**
From the FK image above, derive link lengths, the downturned end-effector orientation (z down, yaw in plane), and implement two functions in `solutions/so101_ik.py`:
- `numerical_ik_so101_downturned`: build the 4×4 target transform for the desired (position, yaw), then solve using `pk.PseudoInverseIK`. Return the best converged joint vector or `None`.
- `so101_downturned_ik_formulas`: return a dict mapping each of the five joint names (see `SO101_JOINT_NAMES`) to a sympy expression in `(x, y, z, yaw)`. A helper `analytical_ik_so101_downturned` lambdifies your formulas for evaluation.

Hidden tests compare your sympy expressions against the reference. Verify with the visualization below and the unit tests. Think about how the exact and numerical solutions behave when self-collisions or unreachable poses are considered.

In [ ]:
from lib.pre_grasp_grid import show_pre_grasp_grid
from solutions.so101_ik import numerical_ik_so101_downturned

PRE_GRASP_POSES = [
    (np.array([0.09, 0.02, -0.13]), 0.39),
    (np.array([0.12, -0.03, -0.12]), -0.56),
    (np.array([0.13, 0.00, -0.15]), 0.09),
    (np.array([0.08, 0.00, -0.08]), 0.27),
    (np.array([0.10, -0.01, -0.12]), 0.11),
    (np.array([0.12, -0.04, -0.13]), -0.60),
]
show_pre_grasp_grid(SO101_URDF, PRE_GRASP_POSES, ik_solver=numerical_ik_so101_downturned)

# Problem 2: Broom Racing (15 pts to Gryffindor)

**Trivia.** 
You are a brave Gryffindor wizard trying out for the Quidditch team.
Currently you are tied with an arrogant Slytherin apprentice for the place in a team, so the committee has set a series of tests to decide who is the better broom handler. 
Magical cruise control keeps your forward speed constant, but riding a broom has limits: you cannot turn too sharply (centrifugal force would throw you off), and you cannot pitch the broom too far up or down (you would slip and fall). 
There are three contests: (1) fly through a Quidditch gate – a ring high in the air, so narrow that only one passing direction is safe; (2) catch the Snitch at a given point in space; (3) catch the ball and carry it through the gate (combination of the first two). 
In each duel you both start at some identical position and orientation in the air and must finish the task faster than your opponent. 
Use your LLMos spell and [pyrseltongue](https://harrypotter.fandom.com/wiki/Parseltongue) to estimate the shortest path. 
You should be equal or better than your opponent on any of the challenges.

**Model.** 
With $v$ as speed (we use units so $v = 1$), $\theta$ the heading angle, $\phi$ the pitch angle, and $u_1$, $u_2$ the control inputs (left–right and up–down steering), the equations of motion are
$$
\begin{cases} 
\dot{x} = v \cos \theta \cos \phi \\
\dot{y} = v \sin \theta \cos \phi \\
\dot{z} = v \sin \phi \\
\dot{\theta} = \frac{u_1}{\cos \phi} \\
\dot{\phi} = u_2 \\
\end{cases}
$$
Safety constraints: 
* pitch is limited by $\phi \in [\phi_{\min}, \phi_{\max}]$, and 
* path curvature is bounded by $\kappa = \sqrt{u_1^2 + u_2^2} \leq \kappa_{\max}$.

For simplicity take $\phi_{\min} = -\phi_{\max} = -45^{\circ}$ and $\kappa_{\max} = 1$.

Task requires you to output $[0, 1]$-parametrized curve $r(s)$ that satisfies equation and constraints listed above under the reparametrization $t(s) = \int_0^s \|\dot{r(\tau)}\| d\tau$ (in other words, $r(0)$ is the starting pose, $r(1)$ is the final pose and everything in between is uniform along the curve's length).

Your opponent tries to follow a heuristic length-optimal path described in [Dubins3d, Vana](https://comrob.fel.cvut.cz/papers/icra20dubins3d.pdf). For tasks where direction of visited configuration is not specified it is chosen by some heuristic rule. It is guaranteed that Euclidean distances between any mandatory configurations is greater than 6.

**Task.**
Implement three functions in `solutions/broom_racing.py`:
- `gate_pass(start, goal) -> curve`: fly from `start` to `goal` (both `Configuration`).
- `catch_snitch(start, goal_xyz) -> curve`: reach `goal_xyz` (`XYZConfiguration`, only position matters).
- `catch_ball_and_gate(start, intermediate_goal, final_goal) -> curve`: reach the ball then fly through the gate.

Each returns a function `curve(s: ndarray) -> Configuration` with $s \in [0, 1]$. See `lib/broom_types.py` for `Configuration`, `XYZConfiguration`, and constraint-checking helpers. Returned function will be tested to satisfy discretized EOM and curvature/pitch constraints. Total path length must be at most equal to reference implementation.

In [ ]:
from lib.broom_types import Configuration, XYZConfiguration, check_all, curve_length
from lib.broom_viz import show_broom_path
from solutions.broom_racing import gate_pass

start = Configuration(0, 0, 0, 0, 0)
goal = Configuration(8, 0, 0, 0, 0)
curve = gate_pass(start, goal)
show_broom_path(curve, start, goal, title="gate_pass example")
ok, errors = check_all(curve, start, goal=goal)
print(f"constraints ok: {ok}  length: {curve_length(curve):.2f}")
if errors:
    print("errors:", errors)

# Problem 3: Optimal WormKnot (10 pts + 5 bonus top-20 leaderboard)

**Trivia.** 
Imagine a really long snake-like robot that has just been used for a very peculiar medical operation. 
Poor thing is tired of doing med stuff, so all it wants is to just roll into a teeny tiny lump and stay silent and cozy like that. 
Show me someone who hasn't experienced this. 
Our duty is to help him. 
How hard can it be?

**Task.**
Consider a robot model whose kinematic tree is just a chain of universal joints with circular joint limit. 
Each link is a cylinder with radius 1 and we ignore self-collisions of consecutive links.
Given a sequence of link lengths, provide an angle configuration that minimizes the diagonal of the axis-aligned bounding box of the robot in 3D space.
Test checks whether there are joint limit violations, robot self-intersections and achieved "compactness" is below that of a heuristic solution.

Implement `optimal_worm_config(link_lengths) -> angles` in `solutions/worm_packing.py`.
- `link_lengths`: `(N,)` array of positive lengths; N links, N−1 universal joints, 2×(N−1) angles.
- Circular limit: for each joint $(\theta_1, \theta_2)$, $\theta_1^2 + \theta_2^2 \leq (\pi/2)^2$.
- Cylinder radius = 1; ignore self-collision for consecutive links.

In [ ]:
from lib.worm_viz import show_worm, segment_endpoints
from solutions.worm_packing import optimal_worm_config

link_lengths = np.array([2.0, 2.0, 2.0])
angles = optimal_worm_config(link_lengths)
display(show_worm(link_lengths, angles, show_aabb=True))
ep = segment_endpoints(link_lengths, angles)
diag = np.linalg.norm(ep.max(axis=0) - ep.min(axis=0))
print(f"AABB diagonal: {diag:.4f}")